# TF-IDF + 로지스틱 회귀 계수 — 국가별 상위/하위 20개 단어

## 목표
`kiwi_형태소분석_4.ipynb`에서 만든 `리뷰_형태소` 컬럼(형태소 분석 완료된, 공백으로 구분된 토큰 문자열)을 사용해
국가별로 TF-IDF(유니그램+바이그램) 벡터를 만들고, 종속변수 `Y`(별점 4·5 → 1, 1·2 → 0)를 예측하는
로지스틱 회귀를 학습합니다. 회귀 계수가 큰 단어일수록 긍정 리뷰(Y=1)와, 계수가 작은(음수) 단어일수록
부정 리뷰(Y=0)와 강하게 연관됩니다.

## 처리 방식
- 대상 파일: `러시아_Wildberries_리뷰_형태소분석.xlsx`, `미국_ULTA_리뷰_형태소분석.xlsx`,
  `일본_라쿠텐_리뷰_형태소분석.xlsx`, `한국_올리브영_리뷰_형태소분석.xlsx` (4개국, 파일명으로 국가 자동 인식)
- `리뷰_형태소`는 이미 형태소 분석이 끝난 공백 구분 토큰이므로, TF-IDF가 이를 **다시 정규식으로 쪼개지 않도록**
  `tokenizer=str.split`로 지정 (sklearn 기본 정규식은 1글자 토큰인 `향`을 다시 삭제해버리기 때문)
- `ngram_range=(1, 2)`로 유니그램·바이그램을 한 번에 학습 (상위/하위 20위 안에 둘 다 섞여서 나옴)
- 국가별로 로지스틱 회귀(`class_weight='balanced'`)를 따로 학습해 계수를 뽑고, 계수 기준 상위 20개(긍정 연관) /
  하위 20개(부정 연관) 단어를 정리
- 결과를 `TF-IDF_국가별_결과.xlsx` 한 파일로 저장 (국가별 시트 + 전체 요약 시트)

## 주의
- 학습에 사용한 정확도는 별도로 떼어 둔 테스트셋이 아닌, 학습에 쓴 데이터 전체에 대한 정확도입니다.
  이번 분석의 목적이 예측 성능이 아니라 "어떤 단어가 긍정/부정과 연관되는가"를 보는 것이므로, 데이터를 나누지 않고
  전체를 학습에 사용했습니다.

In [ ]:
!pip install -q scikit-learn openpyxl

In [ ]:
import pandas as pd
from google.colab import files

# 4개국 '_리뷰_형태소분석.xlsx' 파일을 한 번에 선택해서 업로드하세요.
uploaded = files.upload()
print(list(uploaded.keys()))

## 1. 국가 자동 인식 & 데이터 로드

업로드한 파일명에 아래 키워드가 포함되어 있으면 해당 국가로 인식합니다.
파일명이 달라서 자동 인식이 안 되면 `COUNTRY_KEYWORDS`를 직접 수정하세요.

In [ ]:
COUNTRY_KEYWORDS = {
    '한국_올리브영': ['올리브영', '한국'],
    '미국_ULTA': ['ULTA', '미국'],
    '일본_라쿠텐': ['라쿠텐', '일본'],
    '러시아_Wildberries': ['Wildberries', '러시아'],
}


def match_country(filename: str):
    for country, keywords in COUNTRY_KEYWORDS.items():
        if any(kw in filename for kw in keywords):
            return country
    return None


country_dfs = {}
for filename in uploaded.keys():
    country = match_country(filename)
    if country is None:
        print(f'[경고] 국가를 자동으로 인식하지 못했습니다: {filename} (건너뜀)')
        continue
    df = pd.read_excel(filename)
    country_dfs[country] = df
    print(f'{country:20s} <- {filename}  {df.shape}')

assert len(country_dfs) > 0, '인식된 국가 데이터가 없습니다. COUNTRY_KEYWORDS를 확인하세요.'

## 2. 국가별 TF-IDF + 로지스틱 회귀 학습

- `min_df=3`: 전체 리뷰 중 2회 이하로만 등장한 n-gram은 제외해, 우연히 한두 건에서만 등장해 계수가
  튀는 단어를 걸러냅니다.
- `등장 리뷰 수`: 각 단어(n-gram)가 실제로 몇 건의 리뷰에 등장했는지 함께 보여줘서, 계수만으로 판단하기
  어려운 경우 참고할 수 있게 했습니다.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

TOP_N = 20
MIN_DF = 3


def analyze_country(df: pd.DataFrame, country: str):
    data = df[['리뷰_형태소', 'Y']].copy()
    data['리뷰_형태소'] = data['리뷰_형태소'].fillna('')
    data = data[data['리뷰_형태소'].str.strip() != '']  # 토큰화 후 빈 리뷰 제외
    data = data.dropna(subset=['Y'])

    if data['Y'].nunique() < 2:
        print(f'[{country}] Y 클래스가 하나뿐이라 로지스틱 회귀를 학습할 수 없습니다. 건너뜁니다.')
        return None

    vectorizer = TfidfVectorizer(
        tokenizer=str.split,   # 이미 형태소 분석된 공백 구분 토큰 -> 재정규화하지 않음 (1글자 '향' 등 보존)
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=MIN_DF,
    )
    X = vectorizer.fit_transform(data['리뷰_형태소'])
    y = data['Y'].astype(int)

    model = LogisticRegression(max_iter=2000, class_weight='balanced')
    model.fit(X, y)

    vocab = vectorizer.get_feature_names_out()
    coef = model.coef_[0]
    doc_freq = np.asarray((X > 0).sum(axis=0)).ravel()

    order = np.argsort(coef)
    bottom_idx = order[:TOP_N]     # 계수가 가장 작은(부정 연관) 20개
    top_idx = order[::-1][:TOP_N]  # 계수가 가장 큰(긍정 연관) 20개

    def make_df(idx):
        return pd.DataFrame({
            '순위': range(1, len(idx) + 1),
            '단어(n-gram)': vocab[idx],
            '계수': coef[idx].round(4),
            '등장 리뷰 수': doc_freq[idx],
        })

    top_df = make_df(top_idx)
    bottom_df = make_df(bottom_idx)

    acc = model.score(X, y)
    print(f'[{country}] 리뷰 수: {len(data)}, 어휘 수: {len(vocab)}, 학습 정확도: {acc:.3f}')
    return top_df, bottom_df


results = {}
for country, df in country_dfs.items():
    result = analyze_country(df, country)
    if result is not None:
        results[country] = result

## 3. 결과 미리보기

In [ ]:
for country, (top_df, bottom_df) in results.items():
    print(f'===== {country} : 상위 20 (긍정 연관) =====')
    display(top_df)
    print(f'===== {country} : 하위 20 (부정 연관) =====')
    display(bottom_df)

## 4. 엑셀로 저장

국가별 시트(상위 20 / 하위 20을 나란히 배치)와, 4개국을 한 번에 비교할 수 있는 `요약` 시트를 함께 저장합니다.

In [ ]:
OUTPUT_FILE = 'TF-IDF_국가별_결과.xlsx'

summary_rows = []
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    for country, (top_df, bottom_df) in results.items():
        sheet_df = pd.DataFrame({
            '순위': top_df['순위'],
            '상위20_단어(긍정연관)': top_df['단어(n-gram)'],
            '상위20_계수': top_df['계수'],
            '상위20_등장리뷰수': top_df['등장 리뷰 수'],
            '하위20_단어(부정연관)': bottom_df['단어(n-gram)'],
            '하위20_계수': bottom_df['계수'],
            '하위20_등장리뷰수': bottom_df['등장 리뷰 수'],
        })
        sheet_name = country[:31]  # 엑셀 시트명은 31자 제한
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

        for _, row in top_df.iterrows():
            summary_rows.append({'국가': country, '구분': '상위(긍정연관)', **row.to_dict()})
        for _, row in bottom_df.iterrows():
            summary_rows.append({'국가': country, '구분': '하위(부정연관)', **row.to_dict()})

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_excel(writer, sheet_name='요약', index=False)

print(f'저장 완료: {OUTPUT_FILE}')

from google.colab import files
files.download(OUTPUT_FILE)